# XP Exercises — Summarization Evaluation (Accuracy vs ROUGE)

**Course:** Developers Institute  **Week 8 - Day 4**  
**Author:** Alex Goldbaum

Hands-on summarization evaluation in eight parts:

I.   Setup and library installation.  
II.  Load a (prompt_text → prompt_title) dataset, sample 100 train / 50 test.  
III. Implement `summarize_with_t5(...)` with batching + CUDA management.  
IV.  Compute string-equality **accuracy** — and see why it is useless here.  
V.   Implement **`compute_rouge_score(...)`** with HF `evaluate`.  
VI.  Sanity-check ROUGE behaviour: exact match, null predictions, stemming,
     n-gram differences, symmetry.  
VII. Add `summarize_with_gpt2(...)` (with the *TL;DR:* trick), compute per-row
     ROUGE for **t5-small / t5-base / gpt2**.  
VIII.Aggregate into a single comparison table + side-by-side summary view.

**Dataset note.** The brief references a CSV with `prompt_text` and
`prompt_title` columns. If `train.csv` / `test.csv` are present locally the
notebook uses them; otherwise it falls back to **`cnn_dailymail`** from HF
Datasets, renamed to match the expected schema so the rest of the pipeline
is identical.


## Part I — Setup


In [ ]:
!pip install -q rouge_score==0.1.2 evaluate datasets nltk
!pip install -qU accelerate transformers


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import gc
import os
import numpy as np
import pandas as pd
import torch

import nltk
nltk.download('punkt', quiet=True)
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    pass
from nltk.tokenize import sent_tokenize

import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## Part II — Dataset Loading and Exploration


In [ ]:
TRAIN_CSV = 'train.csv'
TEST_CSV = 'test.csv'

if os.path.exists(TRAIN_CSV) and os.path.exists(TEST_CSV):
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    print(f'Loaded {len(train_df)} train and {len(test_df)} test rows from CSV.')
else:
    print('Local CSV files not found - falling back to cnn_dailymail (HF Datasets).')
    from datasets import load_dataset
    cnn_train = load_dataset('cnn_dailymail', '3.0.0', split='train[:500]')
    cnn_test  = load_dataset('cnn_dailymail', '3.0.0', split='test[:200]')
    train_df = pd.DataFrame({
        'prompt_text': cnn_train['article'],
        'prompt_title': cnn_train['highlights'],
    })
    test_df = pd.DataFrame({
        'prompt_text': cnn_test['article'],
        'prompt_title': cnn_test['highlights'],
    })
    print(f'Loaded {len(train_df)} train and {len(test_df)} test rows from cnn_dailymail.')


In [ ]:
# Sample 100 train / 50 test to keep the notebook fast
train_sample = train_df.sample(n=100, random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=50, random_state=42).reset_index(drop=True)

print('Train sample shape:', train_sample.shape)
print('Test  sample shape:', test_sample.shape)

# Inspect the first training example
print('\n--- First training example ---')
row0 = train_sample.iloc[0]
print('prompt_text (article, first 500 chars):')
print(row0['prompt_text'][:500])
print('\nprompt_title (reference summary):')
print(row0['prompt_title'])


In [ ]:
# Quick view of both DataFrames
print('Train sample head:')
train_sample.head(3)


## Part III — Summarization with T5


In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer, AutoModelForCausalLM


def batch_generator(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def summarize_with_t5(
    texts,
    model_name: str = 't5-small',
    batch_size: int = 8,
    max_input_length: int = 512,
    max_new_tokens: int = 64,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device).eval()

    summaries = []
    with torch.no_grad():
        for batch in batch_generator(list(texts), batch_size):
            prefixed = [f'summarize: {t}' for t in batch]
            enc = tokenizer(
                prefixed,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=max_input_length,
            ).to(device)
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                num_beams=4,
                length_penalty=2.0,
                early_stopping=True,
            )
            for ids in out:
                summaries.append(tokenizer.decode(ids, skip_special_tokens=True))
            # House-keeping after each batch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    # Free the model when we are done
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return summaries


In [ ]:
# Generate summaries with t5-small on the training sample
t5_small_summaries = summarize_with_t5(
    train_sample['prompt_text'].tolist(),
    model_name='t5-small',
    batch_size=8,
)
print(f'Generated {len(t5_small_summaries)} summaries with t5-small.')

results_t5_small = pd.DataFrame({
    'reference': train_sample['prompt_title'],
    't5-small': t5_small_summaries,
})
results_t5_small.head(5)


## Part IV — Accuracy Evaluation


In [ ]:
def accuracy(predictions, references):
    """Strict string equality after whitespace normalization."""
    correct = 0
    for p, r in zip(predictions, references):
        if str(p).strip() == str(r).strip():
            correct += 1
    return correct / max(1, len(predictions))


acc = accuracy(t5_small_summaries, train_sample['prompt_title'].tolist())
print(f'String-equality accuracy of t5-small: {acc:.4f}')


**Why is this 0.0000 (or very close)?** A summary is a free-form paraphrase
of an article — two humans will rarely write the *exact same* summary, let
alone a model and a human. Comparing for **literal equality** misses every
small reword, punctuation difference, or synonym choice. Accuracy is the
wrong metric for generation tasks; we need something that scores
**overlap** rather than equality. That is ROUGE.


## Part V — ROUGE Metric Implementation


In [ ]:
rouge_metric = evaluate.load('rouge')


def _add_newlines(text):
    """Insert sentence-level newlines as the ROUGE library expects."""
    return '\n'.join(sent_tokenize(text.strip()))


def compute_rouge_score(predictions, references, use_stemmer: bool = True,
                       rouge_types=None):
    preds = [_add_newlines(p) for p in predictions]
    refs = [_add_newlines(r) for r in references]
    return rouge_metric.compute(
        predictions=preds,
        references=refs,
        use_stemmer=use_stemmer,
        rouge_types=rouge_types if rouge_types else ['rouge1', 'rouge2', 'rougeL', 'rougeLsum'],
    )


rouge_t5_small = compute_rouge_score(t5_small_summaries, train_sample['prompt_title'].tolist())
print('t5-small ROUGE on the training sample:')
for k, v in rouge_t5_small.items():
    print(f'  {k:>10}: {v:.4f}')


## Part VI — Understanding ROUGE


### 6.1 Exact match (predictions == references)


In [ ]:
refs = train_sample['prompt_title'].tolist()
exact_rouge = compute_rouge_score(refs, refs)
print('Exact-match ROUGE:')
for k, v in exact_rouge.items():
    print(f'  {k:>10}: {v:.4f}')


All scores are 1.0 — when prediction equals reference, every n-gram
overlaps. This is the upper bound.


### 6.2 Null predictions (empty strings)


In [ ]:
null_preds = [''] * len(refs)
null_rouge = compute_rouge_score(null_preds, refs)
print('Null-prediction ROUGE:')
for k, v in null_rouge.items():
    print(f'  {k:>10}: {v:.4f}')


All scores are 0.0 — no overlap is possible when the prediction is empty.


### 6.3 Stemming effect


In [ ]:
small_pred = ['I am running quickly through the garden']
small_ref  = ['I ran quickly through the gardens']

with_stem    = compute_rouge_score(small_pred, small_ref, use_stemmer=True)
without_stem = compute_rouge_score(small_pred, small_ref, use_stemmer=False)

print('With stemming   :', {k: round(v, 4) for k, v in with_stem.items()})
print('Without stemming:', {k: round(v, 4) for k, v in without_stem.items()})


Stemming makes *running / ran* and *garden / gardens* count as the same
token — so scores are higher with `use_stemmer=True`. This is generally
the right default for summarization evaluation.


### 6.4 N-gram analysis — ROUGE-1 vs ROUGE-2


In [ ]:
pairs = [
    ('The cat sat on the mat',  'A cat is sitting on the mat'),
    ('Quick brown fox jumps',    'The quick brown fox leaps over the dog'),
    ('Hello world!',             'Greetings world!'),
]

for pred, ref in pairs:
    r = compute_rouge_score([pred], [ref])
    print(f'PRED: {pred}')
    print(f'REF : {ref}')
    print(f'  rouge1={r["rouge1"]:.3f}  rouge2={r["rouge2"]:.3f}  rougeL={r["rougeL"]:.3f}')
    print()


ROUGE-1 counts single-word overlap; ROUGE-2 counts overlapping bigrams,
which is much stricter — phrases like *quick brown fox* only count when
they appear together in both texts. ROUGE-L measures the longest common
subsequence.


### 6.5 Symmetry of ROUGE


In [ ]:
a = ['The cat sat on the mat']
b = ['A cat is sitting on the mat']

ab = compute_rouge_score(a, b)
ba = compute_rouge_score(b, a)
print('ROUGE(a, b):', {k: round(v, 4) for k, v in ab.items()})
print('ROUGE(b, a):', {k: round(v, 4) for k, v in ba.items()})


ROUGE uses **F-measure** by default (harmonic mean of precision and
recall), which is symmetric — swapping predictions and references gives
essentially the same score. (If you switched to raw recall, you would see
an asymmetry.)


## Part VII — Comparing Small and Large Models


### `summarize_with_gpt2` — using the **TL;DR:** prompt

GPT-2 was not trained for summarization, but the original paper showed it
follows the *TL;DR:* convention reasonably well in zero-shot. We append
*TL;DR:* to the article, truncate so the output has room (GPT-2's context
is only 1024 tokens), generate, and return the text after the marker.


In [ ]:
def summarize_with_gpt2(
    texts,
    model_name: str = 'gpt2',
    batch_size: int = 4,
    max_input_tokens: int = 800,
    max_new_tokens: int = 64,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

    summaries = []
    with torch.no_grad():
        for batch in batch_generator(list(texts), batch_size):
            # Build prompts and aggressively truncate the article side
            prompts = []
            for t in batch:
                # Truncate the article to keep total prompt under max_input_tokens
                article_ids = tokenizer.encode(t, truncation=True, max_length=max_input_tokens - 8)
                article = tokenizer.decode(article_ids, skip_special_tokens=True)
                prompts.append(f'{article}\nTL;DR:')

            enc = tokenizer(
                prompts, return_tensors='pt',
                padding=True, truncation=True, max_length=max_input_tokens,
            ).to(device)

            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                num_beams=1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
            # Strip the prompt off each generation
            for ids, prompt in zip(out, prompts):
                full = tokenizer.decode(ids, skip_special_tokens=True)
                after = full.split('TL;DR:', 1)[-1].strip()
                # Take only the first line / sentence as the summary
                first_line = after.split('\n', 1)[0].strip()
                summaries.append(first_line)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return summaries


In [ ]:
# Generate summaries with all three models
articles = train_sample['prompt_text'].tolist()
references = train_sample['prompt_title'].tolist()

print('Running t5-small...')
summaries_t5_small = t5_small_summaries  # already generated in Part III

print('Running t5-base...')
summaries_t5_base = summarize_with_t5(articles, model_name='t5-base', batch_size=4)

print('Running gpt2...')
summaries_gpt2 = summarize_with_gpt2(articles, model_name='gpt2', batch_size=4)

print('All three models done.')


### Per-row ROUGE scores


In [ ]:
def compute_rouge_per_row(predictions, references):
    rows = []
    for p, r in zip(predictions, references):
        scores = compute_rouge_score([p], [r])
        rows.append({
            'rouge1': scores['rouge1'],
            'rouge2': scores['rouge2'],
            'rougeL': scores['rougeL'],
        })
    return pd.DataFrame(rows)


per_row_small = compute_rouge_per_row(summaries_t5_small, references)
per_row_base  = compute_rouge_per_row(summaries_t5_base, references)
per_row_gpt2  = compute_rouge_per_row(summaries_gpt2, references)

print('Per-row ROUGE — t5-small (first 5):')
print(per_row_small.head(5))
print()
print('Per-row ROUGE — t5-base (first 5):')
print(per_row_base.head(5))
print()
print('Per-row ROUGE — gpt2 (first 5):')
print(per_row_gpt2.head(5))


## Part VIII — Comparing All Models


In [ ]:
def compare_models(model_summaries, references):
    rows = []
    for name, preds in model_summaries.items():
        scores = compute_rouge_score(preds, references)
        rows.append({
            'model': name,
            **{k: round(v, 4) for k, v in scores.items()},
        })
    return pd.DataFrame(rows).set_index('model')


model_summaries = {
    't5-small': summaries_t5_small,
    't5-base':  summaries_t5_base,
    'gpt2':     summaries_gpt2,
}

comparison = compare_models(model_summaries, references)
print('Average ROUGE per model:')
comparison


In [ ]:
def compare_models_summaries(articles, references, model_summaries, n: int = 5):
    df = pd.DataFrame({
        'article_preview': [a[:200] + ('...' if len(a) > 200 else '') for a in articles[:n]],
        'reference': references[:n],
        **{name: preds[:n] for name, preds in model_summaries.items()},
    })
    return df


side_by_side = compare_models_summaries(
    articles, references, model_summaries, n=5,
)
pd.set_option('display.max_colwidth', 250)
side_by_side


## Summary

- **Accuracy is the wrong metric for summarization.** Free-form text
  rarely matches literally, so accuracy collapses to zero and tells us
  nothing about quality.
- **ROUGE** measures n-gram and longest-common-subsequence overlap. ROUGE-1
  is forgiving (single tokens), ROUGE-2 is stricter (bigrams), ROUGE-L
  rewards order-preserving overlap.
- **Stemming** systematically raises scores — and is on by default in our
  helper — because morphological variants (`running`/`ran`) count as one.
- **`t5-small`** is a fast baseline. **`t5-base`** typically beats it on
  every ROUGE variant at the cost of ~3× more parameters. **`gpt2`**, used
  zero-shot with the *TL;DR:* trick, is competitive but inconsistent —
  it was not trained for summarization.
- The full comparison framework (`summarize_with_*`, `compute_rouge_score`,
  `compute_rouge_per_row`, `compare_models`, `compare_models_summaries`)
  generalises to any summarization model — swap in BART, PEGASUS,
  flan-t5-large, or a fine-tuned checkpoint and the workflow is unchanged.
